# Day 1 : Titanic Dataset

Question 1

In [ ]:
import pandas as pd

train = pd.read_csv("train.csv")

print(f"1. Number of Rows =" ,train.shape[0])
print(f"2. Number of Columns =" ,train.shape[1])
print(f"3. Datatypes of each column=" ,train.dtypes)  #i had forgotten it was dtypes and not type/dtype so i searched it on Google
print(f"4. Total memory usage =", train.memory_usage().sum()) #i checked pandas docs to find this

1. Number of Rows = 891
2. Number of Columns = 12
3. Datatypes of each column= PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object
4. Total memory usage = 85668


Question 2

In [162]:
print("1. Columns containing missing values:")
missing = []
for col in train.columns:
    if train[col].isnull().any():
        missing.append(col)
        print(col)

print("2. Missing Count:")
missing_dict = {}
for col in missing:
    missing_count = train[col].isnull().sum()
    missing_percentage = missing_count*100/len(train)
    missing_dict[col] = missing_percentage.round(4)
    print(f"Missing in {col} = {missing_count}")

print("3. Missing Percentage in Sorted Dataframe:")
print(missing_dict)
missing_df = pd.DataFrame(columns = ['col_name','missing_percent'])
missing_df['col_name'] = missing_dict.keys()
missing_df['missing_percent'] = missing_dict.values()
missing_df.sort_values(by='missing_percent', ascending=False) #got this from the docs as well

1. Columns containing missing values:
Age
Cabin
Embarked
2. Missing Count:
Missing in Age = 177
Missing in Cabin = 687
Missing in Embarked = 2
3. Missing Percentage in Sorted Dataframe:
{'Age': 19.8653, 'Cabin': 77.1044, 'Embarked': 0.2245}


,col_name,missing_percent
1,Cabin,77.1044
0,Age,19.8653
2,Embarked,0.2245


Question 3

In [155]:
df = train.copy()
df.dropna(inplace=True)

avg_age = df['Age'].mean()
median_age = df['Age'].median()
sorted_df = df.sort_values(by="Age")
youngest = sorted_df['Age'][:1]
oldest = sorted_df['Age'][-1:]
underage = sorted_df[sorted_df['Age']<18].count()[0]

print(f"1. Average Age = {avg_age}")
print(f"2. Median Age = {median_age}")
print(f"3. Youngest = {youngest}")
print(f"4. Oldest = {oldest}")
print(f"Underage = {underage}")

1. Average Age = 35.6744262295082
2. Median Age = 36.0
3. Youngest = 305    0.92
Name: Age, dtype: float64
4. Oldest = 630    80.0
Name: Age, dtype: float64
Underage = 19


C:\Users\aryan\AppData\Local\Temp\ipykernel_16400\1564041161.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  underage = sorted_df[sorted_df['Age']<18].count()[0]


Question 4

In [187]:
overall = len(df[df['Survived']==1])*100/len(df)
males = df[df['Sex']=='male']
male_rate = len(males[males['Survived']==1])*100/len(males)
females = df[df['Sex']=='female']
female_rate = len(females[females['Survived']==1])*100/len(females)

print(f"1. Overall Survival Rate = {overall}")
print(f"2. Male Survival Rate = {male_rate}")
print(f"3. Female Survival Rate = {female_rate}")

1. Overall Survival Rate = 67.21311475409836
2. Male Survival Rate = 43.1578947368421
3. Female Survival Rate = 93.18181818181819


Question 5

In [ ]:
def survival(series):
    rate = round(len(series[series==1])*100/len(series),2)
    return rate

df.groupby("Pclass").agg(
    ppl_count = ("Name", "count"),
    avg_age = ("Age", "mean"),
    survival_rate = ("Survived", lambda x: survival(x))  #I used some help for .agg and custom argument (i have forgotten almost everything smh)
)

,ppl_count,avg_age,survival_rate
Pclass,,,
1,158,37.591266,67.09
2,15,25.266667,80.00
3,10,21.000000,50.00


Question 6

In [209]:
df["FamilySize"] = df['SibSp'] + df['Parch'] + 1

avg_fam = df['FamilySize'].mean()
largest_fam = df['FamilySize'].max()
fam_survival_rate = df.groupby("FamilySize").agg(
survival_rate = ("Survived", lambda x: round(x.sum()*100/x.count(),2)))

print(f"1. Avg Fam Size = {avg_fam}")
print(f"2. Largest Fam Size = {largest_fam}")
print(f"3. Survival Rate by Fam Size :\n", fam_survival_rate)

1. Avg Fam Size = 1.9398907103825136
2. Largest Fam Size = 6
3. Survival Rate by Fam Size :
             survival_rate
FamilySize               
1                   60.76
2                   72.13
3                   71.43
4                   77.78
5                  100.00
6                   50.00


Question 7

In [ ]:
df2 = train.copy()

print("Initial Null Age Count :", df2['Age'].isnull().sum())
median_pclass = df2.groupby("Pclass").agg(
    median_age = ("Age", "median")
)

df2['Age_Filled'] = df2['Age']
for row in df2['Age_Filled']:
    if row.isna() == True:
        if row.loc("Pclass") == 1:
            row['Age_Filled'].fillna(median_pclass['median_age'][0])  # I dont know how to do this

Initial Null Age Count : 177


AttributeError: 'float' object has no attribute 'isna'

Question 8

In [229]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,1
10,11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S,3
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,1


In [249]:
df_sorted = df.sort_values(by="Fare", ascending=False)
cols = ['Name','Fare','Pclass']
df_sorted['Fare'].rank()

737    182.5
679    182.5
88     179.5
27     179.5
438    179.5
       ...  
715      5.0
75       5.0
872      3.0
806      1.5
263      1.5
Name: Fare, Length: 183, dtype: float64

Im unable to do Questions 8, 9 and 10  :(